# Module 1: 基石算子——代码实战

> 从广播机制到激活函数，从归一化到手算梯度——用代码验证每一个概念。

## 0. 环境准备

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 5)
print(f"PyTorch: {torch.__version__}, NumPy: {np.__version__}")

---
## 1. 数学运算
### 1.1 七种基本二元运算

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a = {a.tolist()}")
print(f"b = {b.tolist()}")
print(f"a + b = {(a + b).tolist()}    # 残差连接: x + F(x)")
print(f"a - b = {(a - b).tolist()}    # 差值")
print(f"a * b = {(a * b).tolist()}    # 门控: gate * value")
print(f"a / b = {(a / b).tolist()}    # 缩放/归一化")
print(f"a ** 2 = {(a ** 2).tolist()}  # 幂: RoPE频率")
print(f"max(a,b) = {torch.maximum(a, b).tolist()}")

### 1.2 广播机制——逐维度对齐

In [ ]:
print("=" * 50)
print("能广播的例子:")

# [3, 1, 4] + [1, 5, 4] -> [3, 5, 4]
x = torch.ones(3, 1, 4)
y = torch.ones(1, 5, 4) * 2
z = x + y
print(f"[3,1,4] + [1,5,4] -> {z.shape}  (预期: [3,5,4])")

# [2, 3] + [3] -> [2, 3]
x = torch.ones(2, 3)
y = torch.tensor([10.0, 20.0, 30.0])
z = x + y
print(f"[2,3] + [3] -> {z.shape}   (预期: [2,3])")
print(f"结果:\n{z}")

# BatchNorm 风格的广播: [C] -> [B, C, H, W]
gamma = torch.tensor([1.0, 2.0, 3.0])           # [C]
x = torch.ones(2, 3, 4, 4)                      # [B, C, H, W]
result = x * gamma.view(1, 3, 1, 1)             # 手动 reshape 触发广播
print(f"\nBN风格广播: [3] -> [2,3,4,4], shape = {result.shape}")

In [ ]:
print("=" * 50)
print("不能广播的例子:")

try:
    x = torch.ones(3, 4)
    y = torch.ones(2, 4)
    z = x + y
except RuntimeError as e:
    print(f"[3,4] + [2,4] -> {e}")

try:
    x = torch.ones(2, 3, 4)
    y = torch.ones(5, 4)
    z = x + y
except RuntimeError as e:
    print(f"[2,3,4] + [5,4] -> {e}")

### 1.3 归约运算

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0]])
print(f"原始张量 [{x.shape[0]}x{x.shape[1]}]:\n{x}\n")

print(f"Sum(dim=1)  = {x.sum(dim=1).tolist()}    # 每行求和")
print(f"Mean(dim=1) = {x.mean(dim=1).tolist()}   # 每行平均 (GAP)")
print(f"Max(dim=1)  = {x.max(dim=1).values.tolist()}   # 每行最大值")
print(f"\n全归约:")
print(f"全局 Sum  = {x.sum().item()}")
print(f"全局 Mean = {x.mean().item()}")

---
## 2. 激活函数——画出每一根曲线

In [ ]:
x = torch.linspace(-4, 4, 500)

activations = {
    'ReLU':      F.relu(x),
    'LeakyReLU': F.leaky_relu(x, negative_slope=0.1),
    'GELU':      F.gelu(x),
    'Swish/SiLU': F.silu(x),
    'Sigmoid':   torch.sigmoid(x),
    'Tanh':      torch.tanh(x),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (name, y) in zip(axes.flatten(), activations.items()):
    ax.plot(x.numpy(), y.numpy(), lw=2, color='#2196F3')
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    ax.axvline(x=0, color='gray', ls='--', alpha=0.5)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)'); ax.grid(True, alpha=0.3)
plt.suptitle('激活函数全景对比', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 把 ReLU、GELU、Swish 放一起对比
x = torch.linspace(-3, 3, 300)
plt.figure(figsize=(12, 5))
plt.plot(x.numpy(), F.relu(x).numpy(),  lw=2, label='ReLU (硬截断)')
plt.plot(x.numpy(), F.gelu(x).numpy(),  lw=2, label='GELU (概率门控)')
plt.plot(x.numpy(), F.silu(x).numpy(),  lw=2, label='Swish/SiLU (自门控)')
plt.axhline(y=0, color='gray', ls='--', alpha=0.5)
plt.axvline(x=0, color='gray', ls='--', alpha=0.5)
plt.xlabel('x', fontsize=13); plt.ylabel('f(x)', fontsize=13)
plt.title('ReLU vs GELU vs Swish —— 从硬截断到软门控', fontsize=15, fontweight='bold')
plt.legend(fontsize=12, loc='upper left'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 关键差异: 负值区
print("负值区行为对比:")
for val in [-3, -1, 0]:
    print(f"  x={val:2d}: ReLU={F.relu(torch.tensor(float(val))).item():6.3f}, "
          f"GELU={F.gelu(torch.tensor(float(val))).item():6.3f}, "
          f"Swish={F.silu(torch.tensor(float(val))).item():6.3f}")

### 2.1 Sigmoid vs Softmax——独立判断 vs 联合决策

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.5, -1.0])

# Sigmoid: 每个元素独立判断
sig = torch.sigmoid(logits)
print(f"原始值:   {logits.tolist()}")
print(f"Sigmoid:  {[f'{x:.3f}' for x in sig.tolist()]}")
print(f"  -> 各值独立在(0,1), 总和={sig.sum():.3f} (不是1!)")

# Softmax: 整个向量联合决策
sfm = F.softmax(logits, dim=0)
print(f"\nSoftmax:  {[f'{x:.3f}' for x in sfm.tolist()]}")
print(f"  -> 各值在(0,1), 总和={sfm.sum():.3f} (一定是1)")

# Temperature 效果
temps = [0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
colors = ['#FF5722', '#2196F3', '#4CAF50', '#FFC107']
for ax, T in zip(axes, temps):
    probs = F.softmax(logits / T, dim=0)
    ax.bar(range(4), probs.numpy(), color=colors)
    ax.set_title(f'Temperature = {T}', fontweight='bold')
    ax.set_ylim(0, 1); ax.set_ylabel('Probability')
plt.suptitle('Temperature 对 Softmax 分布的影响', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("T<1: 分布更尖锐 (更确定) | T=1: 原始分布 | T>1: 分布更平滑 (更随机)")

### 2.2 SwiGLU——现代 LLM MLP 的核心

In [ ]:
# SwiGLU = Swish(x.W_gate) * (x.W_up) . W_down
hidden_dim, intermediate_dim = 8, 24  # 3x

x = torch.randn(1, hidden_dim)
W_gate = torch.randn(hidden_dim, intermediate_dim)
W_up   = torch.randn(hidden_dim, intermediate_dim)
W_down = torch.randn(intermediate_dim, hidden_dim)

gate = F.silu(x @ W_gate)   # Swish 门控信号
up   = x @ W_up              # 信息流
gated = gate * up             # 逐元素门控！
output = gated @ W_down       # 投影回 hidden_dim

print(f"输入:     {x.shape}")
print(f"gate:    Swish(x.W_gate) -> {gate.shape}  (门控信号)")
print(f"up:      x.W_up         -> {up.shape}    (信息流)")
print(f"gated:   gate * up      -> {gated.shape}  (选择性放行!)")
print(f"output:  gated.W_down   -> {output.shape} (最终输出)")
print(f"\n门控信号: min={gate.min():.2f}, max={gate.max():.2f}, mean={gate.mean():.2f}")
print(f"约 {((gate > 0.01) & (gate < 0.99)).float().mean()*100:.0f}% 的值在部分门控状态")

---
## 3. 归一化——手写 BN / LN / RMSNorm

In [ ]:
def my_batchnorm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=(0, 2, 3), keepdim=True)
    var  = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)
    return gamma.view(1,-1,1,1) * (x - mean) / torch.sqrt(var + eps) + beta.view(1,-1,1,1)

def my_layernorm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=(1,2,3), keepdim=True)
    var  = x.var(dim=(1,2,3), keepdim=True, unbiased=False)
    return gamma.view(1,-1,1,1) * (x - mean) / torch.sqrt(var + eps) + beta.view(1,-1,1,1)

def my_rmsnorm(x, gamma, eps=1e-6):
    rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + eps)
    return x / rms * gamma

# 测试
B, C, H, W = 2, 3, 4, 4
x = torch.randn(B, C, H, W)
gamma = torch.ones(C)
beta  = torch.zeros(C)

bn_my = my_batchnorm(x, gamma, beta)
bn_pt = F.batch_norm(x, torch.zeros(C), torch.ones(C), training=True)
print(f"BatchNorm - vs PyTorch: {(bn_my - bn_pt).abs().max():.2e}")

ln_my = my_layernorm(x, gamma, beta)
ln_pt = F.layer_norm(x, [C,H,W], weight=gamma, bias=beta)
print(f"LayerNorm - vs PyTorch: {(ln_my - ln_pt).abs().max():.2e}")

x_seq = torch.randn(2, 10, 64)
rms_my = my_rmsnorm(x_seq, torch.ones(64))
print(f"RMSNorm  - shape: {rms_my.shape}, mean~{rms_my.mean():.4f}")

In [ ]:
# 可视化: 归一化前后分布对比
x_raw = torch.randn(1000) * 3 + 2  # mean=2, std=3

x_ln = (x_raw - x_raw.mean()) / x_raw.std()
rms = torch.sqrt(torch.mean(x_raw ** 2))
x_rms = x_raw / rms

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].hist(x_raw.numpy(), 50, color='#FF5722', alpha=0.7)
axes[0].set_title(f'Original: mean={x_raw.mean():.2f}, std={x_raw.std():.2f}')
axes[1].hist(x_ln.numpy(), 50, color='#2196F3', alpha=0.7)
axes[1].set_title(f'LayerNorm: mean={x_ln.mean():.2f}, std={x_ln.std():.2f}')
axes[2].hist(x_rms.numpy(), 50, color='#4CAF50', alpha=0.7)
axes[2].set_title(f'RMSNorm: RMS~{torch.sqrt(torch.mean(x_rms**2)):.2f}')
plt.suptitle('归一化的效果', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 4. 梯度与反向传播——手算验证

In [ ]:
# 手动 SGD: Loss = (wx - target)^2,  梯度 = 2(wx - target) * x
w = torch.tensor([0.0])
x = torch.tensor([2.0])
target = torch.tensor([3.0])
lr = 0.1
history_w, history_loss = [], []

print("手动 SGD (手算梯度):")
print(f"  初始: w={w.item():.4f}")
for step in range(5):
    y_pred = w * x
    loss = (y_pred - target) ** 2
    grad = 2 * (y_pred - target) * x  # 手算梯度!
    w = w - lr * grad
    history_w.append(w.item())
    history_loss.append(loss.item())
    print(f"  Step {step+1}: w={w.item():.4f}, loss={loss.item():.4f}, grad={grad.item():.4f}")

print(f"\n  理论最优 w = target/x = 1.5")
print(f"  实际收敛 w = {w.item():.4f}")

In [ ]:
# PyTorch autograd 对比
w_auto = torch.tensor([0.0], requires_grad=True)
opt = torch.optim.SGD([w_auto], lr=0.1)

print("PyTorch autograd:")
for step in range(5):
    loss = (w_auto * x - target) ** 2
    opt.zero_grad()
    loss.backward()
    opt.step()
    print(f"  Step {step+1}: w={w_auto.item():.4f}, loss={loss.item():.4f}, "
          f"autograd_grad={w_auto.grad.item():.4f}")

# 可视化收敛
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(history_w, 'o-', lw=2, color='#2196F3', markersize=8)
ax1.axhline(y=1.5, color='red', ls='--', label='Optimal w=1.5')
ax1.set_xlabel('Step'); ax1.set_ylabel('w'); ax1.set_title('Parameter Convergence')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history_loss, 'o-', lw=2, color='#FF5722', markersize=8)
ax2.set_xlabel('Step'); ax2.set_ylabel('Loss'); ax2.set_title('Loss Decrease')
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 死亡 ReLU 演示
torch.manual_seed(42)

class TinyNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.fc1 = nn.Linear(10, 10)
        self.fc2 = nn.Linear(10, 10)
        self.act = activation
    def forward(self, x):
        return self.act(self.fc2(self.act(self.fc1(x)))).sum()

x = -torch.randn(1, 10)  # 全是负数!

for name, act in [("ReLU", nn.ReLU()), ("Swish", nn.SiLU())]:
    model = TinyNet(act)
    model(x).backward()
    gn = model.fc1.weight.grad.norm().item()
    print(f"{name}: fc1.weight.grad L2 norm = {gn:.6f}")
    print(f"  -> {'DEAD! 参数无法更新!' if gn == 0 else 'Alive! 可以继续学习'}")

---
## 5. 硬件执行模型——Roofline 与 FLOPs

In [ ]:
def calc_flops(M, N, K):
    return 2 * M * N * K

def calc_mem(M, N, K, dtype_bytes=4):
    return (M*K + K*N + M*N) * dtype_bytes

def arith_intensity(M, N, K, dtype_bytes=4):
    return calc_flops(M, N, K) / calc_mem(M, N, K, dtype_bytes)

print("Qwen3-0.6B Q投影 GEMM [B, 1024] x [1024, 2048]:")
print(f"  {'B':<8} {'MFLOPs':<12} {'MB':<12} {'AI':<8} {'Bottleneck'}")
print(f"  {'-'*50}")
for B in [1, 4, 32, 256, 512]:
    fl = calc_flops(B, 2048, 1024) / 1e6
    mb = calc_mem(B, 2048, 1024, 2) / 1e6
    ai = arith_intensity(B, 2048, 1024, 2)
    bn = "Compute-bound" if ai > 150 else "Memory-bound"
    print(f"  {B:<8} {fl:>8.1f}   {mb:>8.1f}   {ai:>6.1f}   {bn}")
print(f"\n  A100 roofline 拐点 ≈ 150 FLOPs/Byte")

In [ ]:
# Roofline 可视化
fig, ax = plt.subplots(figsize=(10, 6))
ai_range = np.logspace(-1, 3, 1000)
roofline = np.minimum(np.full_like(ai_range, 312), 2.039 * ai_range)
ax.loglog(ai_range, roofline, 'k-', lw=2)
ax.fill_between(ai_range, 2.039*ai_range, alpha=0.1, color='red')
ax.fill_between(ai_range, np.full_like(ai_range, 312), alpha=0.1, color='blue')

ops = {
    'Add': (0.25, 10, 'red'), 'RMSNorm': (0.5, 15, 'red'),
    'Q-Proj(B=1)': (0.5, 50, 'orange'), 'SDPA Dec': (0.02, 5, 'red'),
    'Q-Proj(B=256)': (150, 280, 'green'), 'MatMul': (200, 300, 'green'),
}
for name, (ai, perf, color) in ops.items():
    ax.scatter([ai], [perf], s=100, c=color, edgecolors='black', zorder=10)
    ax.annotate(name, (ai, perf), textcoords="offset points", xytext=(10, 5), fontsize=10)
ax.set_xlabel('Arithmetic Intensity (FLOPs/Byte)', fontsize=13)
ax.set_ylabel('Performance (TFLOPS)', fontsize=13)
ax.set_title('Roofline Model — A100 (FP16 Tensor Core)', fontsize=15, fontweight='bold')
ax.text(0.5, 200, 'Memory-Bound\n(瓶颈在带宽)', fontsize=11, color='red')
ax.text(50, 200, 'Compute-Bound\n(瓶颈在算力)', fontsize=11, color='blue')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

---
## 6. 优化器——SGD / Momentum / Adam 可视化

In [ ]:
def f(w1, w2):
    return w1**2 + 10 * w2**2  # ill-conditioned

def train(opt_cls, lr, steps=30):
    w = torch.tensor([8.0, 8.0], requires_grad=True)
    opt = opt_cls([w], lr=lr)
    path = [w.detach().clone().numpy()]
    for _ in range(steps):
        opt.zero_grad()
        f(w[0], w[1]).backward()
        opt.step()
        path.append(w.detach().clone().numpy())
    return np.array(path)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
configs = [
    ('SGD', torch.optim.SGD, 0.02),
    ('SGD+Momentum', lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9), 0.02),
    ('Adam', torch.optim.Adam, 0.3),
]
for ax, (name, cls, lr) in zip(axes, configs):
    path = train(cls, lr, 30)
    ax.plot(path[:,0], path[:,1], 'o-', markersize=4, lw=1.5, color='#2196F3')
    ax.scatter([0], [0], s=200, marker='*', color='red', zorder=10, edgecolors='black')
    ax.set_xlim(-10, 10); ax.set_ylim(-10, 10)
    ax.set_xlabel('w1'); ax.set_ylabel('w2')
    ax.set_title(f'{name}\nlr={lr}', fontweight='bold')
    ax.grid(True, alpha=0.3)
plt.suptitle('Optimizer Convergence: f(w1,w2) = w1^2 + 10*w2^2', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("SGD: 震荡 | Momentum: 平滑 | Adam: 自适应, 最快")

---
## 7. 学习率调度——Warmup + Cosine

In [ ]:
def warmup_cosine(step, warmup, total, lr_max=1e-3, lr_min=1e-5):
    if step < warmup:
        return lr_max * step / warmup
    progress = (step - warmup) / (total - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

lrs = [warmup_cosine(s, 100, 1000) for s in range(1000)]

plt.figure(figsize=(12, 4))
plt.plot(lrs, lw=2, color='#2196F3')
plt.axvline(x=100, color='red', ls='--', label='Warmup end (step 100)')
plt.xlabel('Training Step', fontsize=13)
plt.ylabel('Learning Rate', fontsize=13)
plt.title('Warmup + Cosine Annealing (LLM 训练标配)', fontsize=15, fontweight='bold')
plt.legend(fontsize=12); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 8. SIMD——"一次处理多个"的直观演示

In [ ]:
import time

N = 10_000_000
a = np.random.randn(N).astype(np.float32)
b = np.random.randn(N).astype(np.float32)

# 模拟标量 (Python loop, 只跑10万次)
t0 = time.time()
c_s = np.zeros(N, dtype=np.float32)
for i in range(min(N, 100000)):
    c_s[i] = a[i] + b[i]
st = time.time() - t0
sns = st / 100000 * 1e9  # ns per element

# 向量化 (SIMD 模拟)
t0 = time.time()
c_v = a + b
vt = time.time() - t0
vns = vt / N * 1e9

print(f"Python loop (标量): {st*1000:.1f}ms / 100K -> {sns:.1f} ns/element")
print(f"NumPy vec  (SIMD):  {vt*1000:.1f}ms / {N//1_000_000}M -> {vns:.2f} ns/element")
print(f"\n加速比: ~{sns/vns:.0f}x")
print(f"\nCPU: AVX-512 一次处理 16 个 FP32")
print(f"GPU: 一个 warp 同时跑 32 个线程")
print(f"SIMD/SIMT = 并行度 = 免费的性能")

## 总结

| 概念 | 代码验证 | 关键收获 |
|------|---------|--------|
| 广播 | 正反例演示 | 从右对齐, 相等或为1 |
| 激活函数 | 6条曲线对比 | Swish/GELU 负值区优于 ReLU |
| Softmax | Temperature实验 | T<1尖锐, T>1平滑 |
| SwiGLU | 手写微型实现 | gate * up = 选择性放行 |
| 归一化 | 手写 BN/LN/RMSNorm | RMSNorm 省30%计算量 |
| 梯度 | 手算 SGD vs autograd | 梯度=方向, lr=步长 |
| 死亡 ReLU | 全负输入对比实验 | Swish 活, ReLU 死 |
| Roofline | FLOPs vs 带宽计算 | B=1 memory-bound, B=512 compute-bound |
| 优化器 | 不良条件函数路径图 | Adam 自适应每参数学习率 |
| 学习率 | Warmup+Cosine 曲线 | LLM 训练标配 |
| SIMD | loop vs 向量化测速 | 并行度 = 免费的性能, ~1000x加速 |